In [1]:
# Assignment Similarity / Plagiarism Detector

import os
import re
import zipfile

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Load assignment documents / dataset
zip_file = "archive.zip"

# Extract dataset
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("plagiarism_dataset")

folder = "plagiarism_dataset"

# Load pair information
data = pd.read_csv(
    os.path.join(folder, "cheating_dataset.csv")
)

print("Dataset Loaded Successfully")
print("--------------------------------")
print(data.head())

print("\nTotal Document Pairs:", len(data))

# Step 2: Clean and normalize the code
def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove comments
    text = re.sub(
        r'#.*',
        ' ',
        text
    )

    # Remove special characters
    text = re.sub(
        r'[^a-z0-9\s]',
        ' ',
        text
    )

    # Remove extra spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

# Step 3: Load and tokenize documents
documents = {}

for file in os.listdir(folder):

    if file.endswith(".py"):

        file_path = os.path.join(
            folder,
            file
        )

        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            text = f.read()

        # Clean text
        cleaned = clean_text(text)

        # Tokenize
        tokens = cleaned.split()

        documents[file] = {
            "original": text,
            "cleaned": cleaned,
            "tokens": tokens
        }

print("\nNumber of Python Documents:")
print(len(documents))

# Display one example
first_file = list(documents.keys())[0]

print("\nSample Document:")
print(first_file)

print("\nTokens:")
print(
    documents[first_file]["tokens"]
)

# Step 4: Convert documents into TF-IDF vectors
document_names = list(documents.keys())

document_texts = [
    documents[file]["cleaned"]
    for file in document_names
]

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    document_texts
)

print("\nTF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

# Step 5: Calculate cosine similarity
similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("\nCosine Similarity Matrix:")
print(
    pd.DataFrame(
        similarity_matrix,
        index=document_names,
        columns=document_names
    ).round(3)
)

# Step 6: Compare similarity with threshold
threshold = 0.70

print("\nSimilarity Threshold:")
print(threshold * 100, "%")

# Step 7: Identify potentially copied documents
results = []

for _, row in data.iterrows():

    file1 = row["File_1"]
    file2 = row["File_2"]

    label = row["Label"]

    # Get document indexes
    index1 = document_names.index(file1)
    index2 = document_names.index(file2)

    # Get cosine similarity
    similarity = similarity_matrix[
        index1,
        index2
    ]

    # Convert to percentage
    percentage = similarity * 100

    # Compare with threshold
    if similarity >= threshold:
        prediction = "Possible Plagiarism"
    else:
        prediction = "Low Similarity"

    # Actual label
    if label == 1:
        actual = "Cheating"
    else:
        actual = "Not Cheating"

    results.append({

        "File 1":
            file1,

        "File 2":
            file2,

        "Similarity Score":
            round(similarity, 4),

        "Similarity (%)":
            round(percentage, 2),

        "Actual Label":
            actual,

        "Prediction":
            prediction
    })

# Step 8: Generate similarity report
report = pd.DataFrame(
    results
)

print("\n======================================")
print("SIMILARITY REPORT")
print("======================================")

print(
    report.head(20)
)

# Step 9: Display most similar document pairs

ranked_report = report.sort_values(
    by="Similarity (%)",
    ascending=False
).reset_index(
    drop=True
)


print("\n======================================")
print("RANKED DOCUMENT PAIRS")
print("======================================")


print(
    ranked_report.head(20)
)


# =========================================================
# Display potentially copied documents
# =========================================================

print("\n======================================")
print("POTENTIALLY COPIED DOCUMENTS")
print("======================================")


suspicious = ranked_report[
    ranked_report["Similarity (%)"]
    >= threshold * 100
]


if suspicious.empty:

    print(
        "No potentially copied documents found."
    )

else:

    print(
        suspicious[
            [
                "File 1",
                "File 2",
                "Similarity (%)",
                "Actual Label",
                "Prediction"
            ]
        ]
    )


# =========================================================
# Save report
# =========================================================

report.to_csv(
    "Plagiarism_Similarity_Report.csv",
    index=False
)


print("\nSimilarity report saved successfully.")

print(
    "\nPlagiarism Detection Completed Successfully!"
)

Dataset Loaded Successfully
--------------------------------
           File_1          File_2  Label
0  submission1.py  submission2.py      1
1  submission1.py  submission3.py      0
2  submission1.py  submission4.py      1
3  submission1.py  submission5.py      0
4  submission2.py  submission3.py      0

Total Document Pairs: 293

Number of Python Documents:
174

Sample Document:
submission1.py

Tokens:
['def', 'add', 'a', 'b', 'return', 'a', 'b']

TF-IDF Matrix Shape:
(174, 1096)

Cosine Similarity Matrix:
                  submission1.py  submission10.py  submission100.py  \
submission1.py             1.000            0.049             0.017   
submission10.py            0.049            1.000             0.003   
submission100.py           0.017            0.003             1.000   
submission101.py           0.022            0.006             0.296   
submission102.py           0.022            0.006             0.296   
...                          ...              ...          